# Learning Engine Simulation Campaign — all stimulus types

This notebook builds a `LearningEngineCircuitSimulationScanConfig` campaign with every
stimulus type that the Learning Engine simulator accepts (the
`LearningEngineCircuitStimulusUnion`).

**Current-clamp stimuli (absolute amplitude)**

- `ConstantCurrentClampSomaticStimulus`
- `LinearCurrentClampSomaticStimulus`
- `MultiPulseCurrentClampSomaticStimulus`
- `SinusoidalCurrentClampSomaticStimulus`

**Spike stimuli (efferent)**

- `PoissonSpikeStimulus`
- `FullySynchronousSpikeStimulus`
- `SinusoidalPoissonSpikeStimulus`
- `InterSpikeIntervalDistributionSpikeStimulus`
- `SpikeTimeDistributionSpikeStimulus`

The campaign is generated locally against a small SONATA circuit checked into
`examples/data/tiny_circuits/` — no entitycore access required. Each grid coordinate
writes a `simulation_config.json` with `target_simulator: "LearningEngine"` ready to
hand off to the Learning Engine runner.

In [ ]:
from pathlib import Path

import obi_one as obi
from obi_one.scientific.tasks.generate_simulations.config.learning_engine.le_circuit import (
    LearningEngineCircuitSimulationScanConfig,
)

## Load a circuit

We use one of the tiny test circuits shipped with the repo so the notebook is
self-contained. Swap in `obi.CircuitFromID(id_str=...)` to point at an entitycore
circuit when generating a campaign for the cloud runner.

In [ ]:
circuit_path_prefix = Path("../../data/tiny_circuits")
circuit_name = "N_10__top_nodes_dim6"
circuit = obi.Circuit(
    name=circuit_name,
    path=str(circuit_path_prefix / circuit_name / "circuit_config.json"),
)
print(
    f"Circuit '{circuit.name}' with "
    f"{circuit.sonata_circuit.nodes[circuit.default_population_name].size} neurons "
    f"and {circuit.sonata_circuit.edges[circuit.default_edge_population_name].size} synapses"
)

## Build the campaign skeleton

`empty_config()` returns a `LearningEngineCircuitSimulationScanConfig` with all
blocks unset. We fill it in by adding timestamps and neuron sets that the stimuli
will reference.

In [ ]:
sim_duration = 1000.0

sim_conf = LearningEngineCircuitSimulationScanConfig.empty_config()

# Campaign metadata
sim_conf.set(
    obi.Info(
        campaign_name="LE \u2014 all stimulus types",
        campaign_description=(
            "Demonstrates building a Learning Engine simulation campaign with every "
            "stimulus type supported by LearningEngineCircuitStimulusUnion."
        ),
    ),
    name="info",
)

# A single timestamp block reused by every stimulus
timestamps = obi.RegularTimestamps(start_time=0.0, number_of_repetitions=2, interval=500.0)
sim_conf.add(timestamps, name="StimTimestamps")

# Biophysical target neurons + an extrinsic VPM source for spike replay
sim_neurons = obi.AllNeurons()
sim_conf.add(sim_neurons, name="AllNeurons")

vpm_neurons = obi.nbS1VPMInputs(sample_percentage=25)
sim_conf.add(vpm_neurons, name="VPMInput")

## Distributions for distribution-based spike stimuli

`InterSpikeIntervalDistributionSpikeStimulus` and `SpikeTimeDistributionSpikeStimulus`
read their spike statistics from a distribution block.

In [ ]:
# ~20 Hz Poisson via exponential ISIs (scale = 50 ms)
isi_distribution = obi.ExponentialDistribution(scale=50.0, random_seed=42)
sim_conf.add(isi_distribution, name="PoissonInterval")

# Uniform spike-time distribution across each stimulus window
spike_time_distribution = obi.FloatUniformDistribution(low=0.0, high=400.0)
sim_conf.add(spike_time_distribution, name="UniformSpikeTimes")

## Current-clamp stimuli

The four absolute current-clamp stimuli supported by the Learning Engine. We make
`ConstantCurrentClampSomaticStimulus.amplitude` a list so the campaign expands into
multiple grid coordinates.

In [ ]:
sim_conf.add(
    obi.ConstantCurrentClampSomaticStimulus(
        timestamps=timestamps.ref,
        neuron_set=sim_neurons.ref,
        duration=100.0,
        amplitude=[0.1, 0.3],  # scan parameter → 2 coordinates
    ),
    name="ConstantCurrent",
)

sim_conf.add(
    obi.LinearCurrentClampSomaticStimulus(
        timestamps=timestamps.ref,
        neuron_set=sim_neurons.ref,
        duration=100.0,
        amplitude_start=0.0,
        amplitude_end=0.3,
    ),
    name="LinearCurrent",
)

sim_conf.add(
    obi.MultiPulseCurrentClampSomaticStimulus(
        timestamps=timestamps.ref,
        neuron_set=sim_neurons.ref,
        duration=300.0,
        amplitude=0.2,
        width=5.0,
        frequency=10.0,
    ),
    name="MultiPulseCurrent",
)

sim_conf.add(
    obi.SinusoidalCurrentClampSomaticStimulus(
        timestamps=timestamps.ref,
        neuron_set=sim_neurons.ref,
        duration=300.0,
        maximum_amplitude=0.15,
        frequency=8.0,
        dt=0.025,
    ),
    name="SinusoidalCurrent",
)

## Spike stimuli

All five spike-stimulus types accepted by the Learning Engine. The VPM source
neuron set provides extrinsic input; spike replay files are generated locally per
coordinate and written as SONATA `_spikes.h5` files alongside `simulation_config.json`.

In [ ]:
sim_conf.add(
    obi.PoissonSpikeStimulus(
        timestamps=timestamps.ref,
        source_neuron_set=vpm_neurons.ref,
        targeted_neuron_set=sim_neurons.ref,
        duration=400.0,
        frequency=10.0,
        random_seed=0,
    ),
    name="PoissonSpikes",
)

sim_conf.add(
    obi.FullySynchronousSpikeStimulus(
        timestamps=timestamps.ref,
        source_neuron_set=vpm_neurons.ref,
        targeted_neuron_set=sim_neurons.ref,
    ),
    name="SynchronousSpikes",
)

sim_conf.add(
    obi.SinusoidalPoissonSpikeStimulus(
        timestamps=timestamps.ref,
        source_neuron_set=vpm_neurons.ref,
        targeted_neuron_set=sim_neurons.ref,
        duration=400.0,
        minimum_rate=0.5,
        maximum_rate=15.0,
        modulation_frequency_hz=4.0,
        phase_degrees=0.0,
        random_seed=0,
    ),
    name="SinusoidalPoissonSpikes",
)

sim_conf.add(
    obi.InterSpikeIntervalDistributionSpikeStimulus(
        timestamps=timestamps.ref,
        source_neuron_set=vpm_neurons.ref,
        targeted_neuron_set=sim_neurons.ref,
        duration=400.0,
        distribution=isi_distribution.ref,
        resample_each_repetition=True,
    ),
    name="ISIDistributionSpikes",
)

sim_conf.add(
    obi.SpikeTimeDistributionSpikeStimulus(
        timestamps=timestamps.ref,
        source_neuron_set=vpm_neurons.ref,
        targeted_neuron_set=sim_neurons.ref,
        duration=400.0,
        mean_firing_rate=10.0,
        distribution=spike_time_distribution.ref,
        resample_each_repetition=True,
    ),
    name="SpikeTimeDistributionSpikes",
)

## Initialize and validate

The Initialize block pins the campaign to a circuit and a simulation length. The
`random_seed` field is hidden in the UI but still scannable from Python.

In [ ]:
sim_conf.set(
    LearningEngineCircuitSimulationScanConfig.Initialize(
        circuit=circuit,
        node_set=sim_neurons.ref,
        simulation_length=sim_duration,
        v_init=-80.0,
        random_seed=1,
    ),
    name="initialize",
)

validated_sim_conf = sim_conf.validated_config()
print(validated_sim_conf)

## Generate the grid scan

Only `ConstantCurrent.amplitude` is a list, so the grid expands into 2 coordinates.
Each coordinate writes a `simulation_config.json`, the staged SONATA circuit and
any required spike-replay HDF5 files.

In [ ]:
output_root = "../../../../obi-output/learning_engine_simulations/grid_scan"

grid_scan = obi.GridScanGenerationTask(
    form=validated_sim_conf,
    coordinate_directory_option="ZERO_INDEX",
    output_root=output_root,
)
grid_scan.multiple_value_parameters(display=True)
grid_scan.execute()
obi.run_task_for_single_configs(single_configs=grid_scan.single_configs)

## Inspect a generated `simulation_config.json`

The Learning Engine config carries `target_simulator: "LearningEngine"` and the
same SONATA `run` / `inputs` structure that the Learning Engine runner expects.

In [ ]:
import json

simulation_config_path = (
    grid_scan.single_configs[0].coordinate_output_root / "simulation_config.json"
)
print(f"simulation_config: {simulation_config_path}\n")

with simulation_config_path.open() as f:
    generated = json.load(f)

print(f"target_simulator: {generated['target_simulator']}")
print(f"run:              {generated['run']}")
print(f"conditions:       {generated['conditions']}")
print(f"inputs:           {list(generated.get('inputs', {}).keys())}")